# Overlapping generations (growth)

Juveniles and adults with separate fitness surfaces, fitness-dependent mortality and
replacement of the dead by offspring of surviving adults (`?EvoMod.Growth`).
Generations here are time steps.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using EvoMod, EvoMod.Growth
using Gadfly, DataFrames, LinearAlgebra, Random, Statistics
include("plots.jl")
Random.seed!(42)

In [ ]:
# Parameters
n_e  = 2000
m    = 400
p    = 4
mu   = 0.0005
mu_b = 0.0001
mu_σ = 0.02
maturity = 50   # age after which an individual is an adult

omega_var = 10
delta_theta_speed = 1/1000

burnin_generations = [500, 500]     # Drift, Stab
direct_generations = [2000, 1000]   # Direct, Stab
thin = 1

ind = Individual(m, p, mu, mu_b, mu_σ, maturity)

In [ ]:
diag_omega = omega_var * Matrix(1.0I, p, p)
burnin_pop = run_pop(ind, n_e, ["Drift", "Stab"], burnin_generations; a_omega = diag_omega, thin = thin)

In [ ]:
delta_theta = delta_theta_speed * [ones(p ÷ 2); -ones(p ÷ 2)]
a_omega = twoModuleMatrix(omega_var * ones(p), 0.8, 0.8)
j_omega = twoModuleMatrix(10 * omega_var * ones(p), 0.8, 0.8)

direct_pop = run_pop(copy(burnin_pop), ["Direct", "Stab"], direct_generations;
                     a_delta_theta = delta_theta, a_omega = a_omega,
                     j_delta_theta = delta_theta, j_omega = j_omega, thin = thin)
pop = direct_pop

In [ ]:
function plotClasses(pop, key_a, key_j; scale = 1)
    r = 1:length(pop.moments)
    df = DataFrame(adults = scale * [pop.moments[i][key_a] for i in r],
                   juveniles = scale * [pop.moments[i][key_j] for i in r],
                   gen = [pop.moments[i]["gen"] for i in r])
    plot(df, layer(y = :adults, x = :gen, Geom.line, Theme(default_color = colorant"red")),
             layer(y = :juveniles, x = :gen, Geom.line),
         Guide.manual_color_key("Legend", ["Adults", "Juveniles"], ["red", "deepskyblue"]))
end
plotClasses(pop, "p_d_adults", "p_d_juveniles"; scale = 100)   # % dying per time step

In [ ]:
plotClasses(pop, "a_fitness", "j_fitness")   # standard deviation of fitness

In [ ]:
plotClasses(pop, "adults", "juveniles")      # proportion of each class

In [ ]:
plotCorrHeatmap(pop.moments[end]["corrP"])

In [ ]:
plotCorrelations(pop; key = "corrP")

In [ ]:
plotTraits(pop)

In [ ]:
plotSurfacePop(pop, pop.a_surface)